# Real-subject MP150 pipeline

This notebook is intentionally thin: set variables here, call the `.py` pipeline functions, and display tables/plots.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

if (Path.cwd() / 'real_subject').exists():
    REAL_SUBJECT_DIR = Path.cwd() / 'real_subject'
else:
    REAL_SUBJECT_DIR = Path.cwd()

if str(REAL_SUBJECT_DIR) not in sys.path:
    sys.path.insert(0, str(REAL_SUBJECT_DIR))

from acquisition import AcquisitionConfig, collect_training_segments
from preprocessing import AudioLabelConfig, PreprocessConfig, preprocess_many_recordings, preprocess_recording
from training import TrainingConfig, WindowConfig, train_validate_pipeline
from testing import collect_preprocess_and_test_trial, predict_labeled_recording
from plots import plot_labeled_recording, plot_predictions_overlay

print('Using real_subject directory:', REAL_SUBJECT_DIR)

## User settings

In [ ]:
RAW_DIR = REAL_SUBJECT_DIR / 'data' / 'raw'
LABELED_DIR = REAL_SUBJECT_DIR / 'data' / 'labeled'
RUN_DIR = REAL_SUBJECT_DIR / 'runs' / 'run_001'

COLLECT_TRAINING = False
TRAIN_SEGMENT_SEC = 300.0
TRAIN_SEGMENT_NAMES = ('train_01', 'train_02')
TRAIN_RAW_NPZ = []

COLLECT_TEST_TRIAL = False
TEST_TRIAL_SEC = 60.0
TEST_TRIAL_NAME = 'test_trial_01'
TEST_RAW_NPZ = None
TEST_LABELED_NPZ = None

ACQ = AcquisitionConfig(
    samplerate=200,
    channels=(1, 2, 3, 4, 5),
    chunk_sec=0.10,
)

PRE = PreprocessConfig(
    eeg_channels=(1, 2, 3, 4),
    audio_channel=5,
    bandpass_low_hz=1.0,
    bandpass_high_hz=40.0,
    filter_order=4,
    demean_channels=True,
)

LABELS = AudioLabelConfig(
    class_names=('Idle', 'Task'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    label_duration_sec=5.0,
    label_start_offset_sec=0.0,
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

WIN = WindowConfig(
    feature_mode='raw_signal',
    window_sec=1.0,
    stride_sec=0.10,
    label_mode='endpoint',
    bandpower_hz=(18.0, 22.0),
)

TRAIN = TrainingConfig(
    train_fraction=0.80,
    hidden_size=32,
    num_layers=1,
    dropout=0.0,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

## Collection

In [ ]:
if COLLECT_TRAINING:
    train_raw_paths = collect_training_segments(
        output_dir=RAW_DIR,
        segment_names=TRAIN_SEGMENT_NAMES,
        config=ACQ,
        duration_sec=TRAIN_SEGMENT_SEC,
    )
else:
    train_raw_paths = [Path(p) for p in TRAIN_RAW_NPZ]

if not train_raw_paths:
    raise ValueError('Set COLLECT_TRAINING=True or populate TRAIN_RAW_NPZ with existing raw .npz files.')

train_raw_paths

## Preprocessing and audio-cue labels

In [ ]:
labeled_paths = preprocess_many_recordings(
    raw_npz_paths=train_raw_paths,
    output_dir=LABELED_DIR,
    preprocess_config=PRE,
    label_config=LABELS,
)

labeled_paths

In [ ]:
fig, ax = plot_labeled_recording(labeled_paths[0], max_duration_sec=30.0)

## Training and validation

In [ ]:
train_result = train_validate_pipeline(
    labeled_npz_paths=labeled_paths,
    output_dir=RUN_DIR,
    window_config=WIN,
    training_config=TRAIN,
)

checkpoint_path = train_result['checkpoint_path']
checkpoint_path

In [ ]:
history = train_result['history']
display(history.tail())
display(train_result['validation_summary'])
display(train_result['validation_per_class'])

ax = history.plot(x='epoch', y=['train_acc', 'val_acc'], marker='o', figsize=(8, 4))
ax.set_ylabel('Accuracy')
ax.set_title('Training and validation accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()

## Testing on an arbitrary-length trial

In [ ]:
if COLLECT_TEST_TRIAL:
    test_result = collect_preprocess_and_test_trial(
        output_dir=RUN_DIR / 'test_trials',
        checkpoint_path=checkpoint_path,
        acquisition_config=ACQ,
        preprocess_config=PRE,
        label_config=LABELS,
        duration_sec=TEST_TRIAL_SEC,
        trial_name=TEST_TRIAL_NAME,
    )
    test_labeled_path = test_result['labeled_path']
elif TEST_LABELED_NPZ is not None:
    test_labeled_path = Path(TEST_LABELED_NPZ)
    test_result = predict_labeled_recording(
        labeled_npz=test_labeled_path,
        checkpoint_path=checkpoint_path,
        output_dir=RUN_DIR / 'test_trials',
    )
elif TEST_RAW_NPZ is not None:
    test_raw_path = Path(TEST_RAW_NPZ)
    test_labeled_path = LABELED_DIR / f'{test_raw_path.stem}_labeled.npz'
    preprocess_recording(test_raw_path, test_labeled_path, PRE, LABELS)
    test_result = predict_labeled_recording(
        labeled_npz=test_labeled_path,
        checkpoint_path=checkpoint_path,
        output_dir=RUN_DIR / 'test_trials',
    )
else:
    raise ValueError('Set COLLECT_TEST_TRIAL=True, TEST_RAW_NPZ, or TEST_LABELED_NPZ.')

display(test_result['summary'])
display(test_result['per_class'])
display(test_result['delay_summary'])

In [ ]:
fig, ax = plot_predictions_overlay(
    labeled_npz=test_labeled_path,
    predictions=test_result['predictions'],
    max_duration_sec=None,
)